# Flow recommender / ranking evaluation

Reframes the accuracy predictor as a **per-task flow ranker**: for each held-out CC18 task,
rank the candidate flows by predicted accuracy and measure how good the recommended flow is.

Protocol: leave-tasks-out (`GroupKFold` on `task_id`) — the same honest split as the main study.
The (task, flow) matrix is sparse, so a task's candidate set is the flows with a known accuracy
on it, and regret is measured against the best true accuracy within that set.

Methods compared: **model** (the predictor), **global_default** (portfolio baseline — flow that is
best on average across training tasks), and **random**. Beating `global_default` is the real claim.

This notebook is thin orchestration over `openml_flow.run_recommender_evaluation`.

## Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import openml_flow as wf

# --- Configuration -----------------------------------------------------------
CACHE_DIR    = "minimal_cache_cc18"          # DiskCache (OpenML bundle + feature matrices)
FLOWS_CACHE  = "minimal_cache_cc18/flows.joblib"
RESULTS_DIR  = "results_recommender_max"     # ranking CSVs land here
FIG_DIR      = Path(RESULTS_DIR) / "figures"

AGG_MODE     = "max"        # rank "which flow can win on this task"; use "mean" for typical perf
MODEL_NAME   = "hist_gbrt"  # ranking-first uses the strongest single tree model
CV_FOLDS     = 5
KS           = (1, 3, 5, 10)
N_RANDOM_SHUFFLES = 20

# Representations to compare. meta_only cannot rank within-task (all flows on a task share
# one feature vector) — kept as a floor that isolates what the flow text adds.
CONFIGS = [
    {"name": "meta_only",  "text_mode": "none",  "use_task_metafeatures": True},
    {"name": "tfidf_only", "text_mode": "tfidf", "use_task_metafeatures": False},
    {"name": "tfidf_meta", "text_mode": "tfidf", "use_task_metafeatures": True},
    # {"name": "minilm_meta", "text_mode": "minilm", "use_task_metafeatures": True},
]

FIG_DIR.mkdir(parents=True, exist_ok=True)
print("configs:", [c["name"] for c in CONFIGS])

## Load data (OpenML bundle + flow records)

Cached after the first download. Set `OPENML_API_KEY` in the environment if needed.

In [ ]:
bundle = wf.load_cc18_from_openml(
    function="predictive_accuracy",
    suite_name="OpenML-CC18",
    cache=wf.DiskCache(CACHE_DIR),
)
flows = wf.load_or_cache_flows(bundle["evals_df"], FLOWS_CACHE)

print("tasks:", bundle["tasks_df"].shape,
      "| eval rows:", bundle["evals_df"].shape,
      "| flows:", len(flows))

## Run the recommender evaluation

Writes `ranking_metrics.csv` (means over tasks) and `ranking_metrics_per_task.csv` into `RESULTS_DIR`.

In [ ]:
rec = wf.run_recommender_evaluation(
    flows=flows,
    tasks_df=bundle["tasks_df"],
    evals_df=bundle["evals_df"],
    configs=CONFIGS,
    agg_mode=AGG_MODE,
    model_name=MODEL_NAME,
    cv_folds=CV_FOLDS,
    ks=KS,
    n_random_shuffles=N_RANDOM_SHUFFLES,
    cache_dir=CACHE_DIR,
    results_dir=RESULTS_DIR,
)

summary   = rec["summary"]
per_task  = rec["per_task"]
print("results dir:", rec["results_dir"])
summary

## Headline table: model vs baselines

Lower `nregret@1` is better (0 = always recommends the best flow); higher `hit@k`, `ndcg`, `spearman` are better.

In [ ]:
show_cols = ["experiment", "method", "n_tasks", "spearman_mean",
             "nregret@1_mean", "nregret@3_mean", "hit@1_mean", "hit@3_mean", "ndcg@5_mean"]
table = summary[show_cols].copy()
method_order = {"model": 0, "global_default": 1, "random": 2}
table = table.sort_values(
    ["experiment", "method"], key=lambda s: s.map(method_order) if s.name == "method" else s
).reset_index(drop=True)
table.round(4)

## Figure 1 — normalized regret@1 by representation and method

The recommender is useful when the **model** bar sits clearly below **global_default**.

In [ ]:
methods = ["model", "global_default", "random"]
colors  = {"model": "#2c7fb8", "global_default": "#7fcdbb", "random": "#d9d9d9"}
exps    = [c["name"] for c in CONFIGS if c["name"] in set(summary["experiment"])]

piv = summary.pivot_table(index="experiment", columns="method", values="nregret@1_mean")
piv = piv.reindex(index=exps, columns=methods)

x = np.arange(len(exps))
w = 0.26
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, m in enumerate(methods):
    ax.bar(x + (i - 1) * w, piv[m].values, width=w, label=m, color=colors[m], edgecolor="black", linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(exps)
ax.set_ylabel("normalized regret@1  (lower = better)")
ax.set_title(f"Zero-shot flow recommendation — {MODEL_NAME}, agg={AGG_MODE}")
ax.legend(title="method", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "regret_at_1.png", dpi=150)
fig.savefig(FIG_DIR / "regret_at_1.pdf")
plt.show()

## Figure 2 — hit@k curves

Probability the truly-best flow is within the top-k recommended, as a function of k.

In [ ]:
fig, axes = plt.subplots(1, len(exps), figsize=(4.2 * len(exps), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, exp in zip(axes, exps):
    sub = summary[summary["experiment"] == exp]
    for m in methods:
        row = sub[sub["method"] == m]
        if row.empty:
            continue
        ys = [float(row[f"hit@{k}_mean"].iloc[0]) for k in KS]
        ax.plot(KS, ys, marker="o", label=m, color=colors[m])
    ax.set_title(exp)
    ax.set_xlabel("k")
    ax.set_xticks(list(KS))
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("hit@k  (P best flow in top-k)")
axes[-1].legend(title="method", frameon=False)
fig.suptitle(f"hit@k — {MODEL_NAME}, agg={AGG_MODE}", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "hit_at_k.png", dpi=150, bbox_inches="tight")
fig.savefig(FIG_DIR / "hit_at_k.pdf", bbox_inches="tight")
plt.show()

## Per-task spread

Regret is an average over tasks; the per-task distribution shows whether the model helps everywhere
or is carried by a few easy tasks. Uses `nregret@1` from `per_task` for the best representation.

In [ ]:
best_exp = (
    summary[summary["method"] == "model"]
    .sort_values("nregret@1_mean").iloc[0]["experiment"]
)
print("best representation (model, lowest nregret@1):", best_exp)

sub = per_task[(per_task["experiment"] == best_exp) & (per_task["method"].isin(["model", "global_default"]))]
fig, ax = plt.subplots(figsize=(7, 4))
data = [sub[sub["method"] == m]["nregret@1"].dropna().values for m in ["model", "global_default"]]
ax.boxplot(data, labels=["model", "global_default"], showmeans=True)
ax.set_ylabel("nregret@1 per task")
ax.set_title(f"Per-task regret spread — {best_exp}")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "per_task_regret_box.png", dpi=150)
plt.show()

## Warm-start: how much search does the ranking save?

Treat each method's ranking as a greedy search: evaluate flows top-down and stop when within
`warm_start_eps` accuracy of the best. `trials_to_target` is how many flow evaluations that takes.
The speedup vs `random` is the headline "AutoML search saved" number.

In [ ]:
tt = rec["trials_to_target"]
cols = ["experiment", "method", "trials_to_target_mean", "trials_to_target_median", "max_trials", "n_tasks"]
tt_show = tt[cols].sort_values(
    ["experiment", "method"],
    key=lambda s: s.map({"model": 0, "global_default": 1, "random": 2}) if s.name == "method" else s,
).reset_index(drop=True)

# speedup = random trials / method trials, per experiment
piv_tt = tt.pivot_table(index="experiment", columns="method", values="trials_to_target_mean")
speedup = pd.DataFrame({
    "model_speedup_vs_random": piv_tt["random"] / piv_tt["model"],
    "model_speedup_vs_default": piv_tt["global_default"] / piv_tt["model"],
}).round(2)
print("trials to within {:.1%} of best (fewer = better):".format(rec["trials_to_target"]["eps_abs"].iloc[0]))
display(tt_show.round(3))
print("\nspeedup (x fewer trials):")
display(speedup)

## Figure 3 — warm-start regret curves

Mean normalized regret as a function of the number of flows evaluated. A good ranker's curve
drops to ~0 far faster than `random`.

In [ ]:
curve = rec["warm_start_curve"]
fig, axes = plt.subplots(1, len(exps), figsize=(4.4 * len(exps), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, exp in zip(axes, exps):
    sub = curve[curve["experiment"] == exp]
    for m in methods:
        cm = sub[sub["method"] == m].sort_values("trial")
        if cm.empty:
            continue
        ax.plot(cm["trial"], cm["nregret_mean"], marker=".", label=m, color=colors[m])
    ax.set_title(exp)
    ax.set_xlabel("flows evaluated (trials)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("mean normalized regret")
axes[-1].legend(title="method", frameon=False)
fig.suptitle(f"Warm-start regret vs #trials \u2014 {MODEL_NAME}, agg={AGG_MODE}", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "warm_start_curve.png", dpi=150, bbox_inches="tight")
fig.savefig(FIG_DIR / "warm_start_curve.pdf", bbox_inches="tight")
plt.show()

## Notes

- `meta_only` is expected to underperform: with no flow-level features every flow on a task shares
  one feature vector, so the model cannot rank within-task. It isolates what the flow text adds.
- Switch `AGG_MODE` to `"mean"` to rank by typical rather than best-case accuracy.
- Outputs written to `RESULTS_DIR`: `ranking_metrics.csv`, `ranking_metrics_per_task.csv`,
  `warm_start_curve.csv`, `warm_start_trials_to_target.csv` (+ figures under `figures/`).